Step 1. Silver Listings

Read Data

In [0]:
from pyspark.sql.functions import col

df_listings = spark.read.format("delta").load(
    "/Volumes/workspace/default/airbnb_bronze_2/listings"
)

In [0]:
print(df_listings.columns)

['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'calendar_updated', 'has_availability', 'availability_30', 'availability_60', 'availability_90', 'availabil

Clean Important Columns For Analysis

In [0]:
from pyspark.sql.functions import regexp_replace, trim, lower

df_listings = spark.read.format("delta").load(
    "/Volumes/workspace/default/airbnb_bronze_2/listings"
)

df_listings_silver = df_listings.select(
    col("id").cast("long").alias("listing_id"),
    col("name"),
    col("host_id").cast("long"),
    col("host_name"),
    col("host_is_superhost"),
    col("neighbourhood"),
    col("neighbourhood_cleansed"),
    col("neighbourhood_group_cleansed"),
    col("latitude").cast("double"),
    col("longitude").cast("double"),
    col("property_type"),
    col("room_type"),
    col("accommodates").cast("int"),
    col("bathrooms"),
    col("bathrooms_text"),
    col("bedrooms").cast("double"),
    col("beds").cast("double"),
    col("amenities"),
    regexp_replace(col("price"), "[$,]", "").cast("double").alias("price"),
    col("minimum_nights").cast("int"),
    col("maximum_nights").cast("int"),
    col("availability_30").cast("int"),
    col("availability_60").cast("int"),
    col("availability_90").cast("int"),
    col("availability_365").cast("int"),
    col("number_of_reviews").cast("int"),
    col("number_of_reviews_ltm").cast("int"),
    col("number_of_reviews_l30d").cast("int"),
    col("review_scores_rating").cast("double"),
    col("review_scores_accuracy").cast("double"),
    col("review_scores_cleanliness").cast("double"),
    col("review_scores_checkin").cast("double"),
    col("review_scores_communication").cast("double"),
    col("review_scores_location").cast("double"),
    col("review_scores_value").cast("double"),
    col("instant_bookable"),
    col("calculated_host_listings_count").cast("int"),
    col("reviews_per_month").cast("double"),
    col("estimated_occupancy_l365d").cast("double"),
    col("estimated_revenue_l365d").cast("double"),
    col("city"),
    col("source_file"),
    col("ingestion_timestamp")
).filter(
    col("price").isNotNull()
).filter(
    col("price") > 0
).withColumn(
    "room_type", trim(lower(col("room_type")))
).dropDuplicates(
    ["listing_id", "city"]
)


Write Silver Listings

In [0]:
df_listings_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/airbnb_silver_2/listings")

Validation

In [0]:
display(df_listings_silver.limit(10))

listing_id,name,host_id,host_name,host_is_superhost,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,reviews_per_month,estimated_occupancy_l365d,estimated_revenue_l365d,city,source_file,ingestion_timestamp
78884,Cozy Barton Hills Studio | Walk to Zilker Park,2157243,Sean,t,Neighborhood highlights,78704,null,30.26126,-97.77358,Entire rental unit,entire home/apt,2,1.0,1 bath,0.0,2.0,"[""Private patio or balcony"", ""Stove"", ""First aid kit"", ""Bathtub"", ""Iron"", ""Ceiling fan"", ""Dishwasher"", ""Hot water"", ""Shower gel"", ""Dishes and silverware"", ""Conditioner"", ""Refrigerator"", ""Freezer"", ""Paid washer \u2013 In building"", ""Self check-in"", ""Free parking on premises"", ""Keypad"", ""Wifi"", ""Heating"", ""Hair dryer"", ""Bed linens"", ""Cooking basics"", ""TV"", ""Long term stays allowed"", ""Window AC unit"", ""Oven"", ""Smoke alarm"", ""Essentials"", ""Luggage dropoff allowed"", ""Paid dryer \u2013 In building"", ""Dedicated workspace"", ""Fire extinguisher"", ""Shampoo"", ""Body soap"", ""Extra pillows and blankets"", ""Coffee maker"", ""Free street parking"", ""Carbon monoxide alarm"", ""Microwave"", ""Kitchen"", ""Hangers""]",103.0,2,90,18,44,74,150,219,14,1,4.79,4.82,4.87,4.95,4.95,4.97,4.78,t,9,1.24,84.0,8652.0,austin,austin_listings.csv,2026-04-13T02:00:43.225Z
89475,"2 Living Rooms, Lots of Big Beds! 3 king, 2 queen",487072,Gregory,t,Neighborhood highlights,78757,null,30.35681,-97.72618,Entire home,entire home/apt,8,2.0,2 baths,4.0,6.0,"[""Stove"", ""First aid kit"", ""Bathtub"", ""TV with standard cable"", ""Iron"", ""Wine glasses"", ""Blender"", ""Private entrance"", ""Outdoor furniture"", ""Pets allowed"", ""Dishwasher"", ""High chair"", ""Hot water"", ""Dishes and silverware"", ""Conditioner"", ""Refrigerator"", ""Freezer"", ""Self check-in"", ""Dining table"", ""Free parking on premises"", ""Keypad"", ""Heating"", ""Hair dryer"", ""Bed linens"", ""Cooking basics"", ""Long term stays allowed"", ""Baking sheet"", ""Children\u2019s dinnerware"", ""Patio or balcony"", ""Oven"", ""Children\u2019s books and toys"", ""Dryer"", ""Pack \u2019n play/Travel crib"", ""Smoke alarm"", ""Washer"", ""Essentials"", ""Ethernet connection"", ""Books and reading material"", ""Air conditioning"", ""Exterior security cameras on property"", ""Dedicated workspace"", ""Fire extinguisher"", ""Fast wifi \u2013 414 Mbps"", ""Barbecue utensils"", ""BBQ grill"", ""Crib"", ""Shampoo"", ""Body soap"", ""Extra pillows and blankets"", ""Clothing storage"", ""Coffee maker"", ""Carbon monoxide alarm"", ""Toaster"", ""Microwave"", ""Free street parking"", ""Kitchen"", ""Backyard"", ""Hangers""]",167.0,45,999,0,14,44,319,129,4,1,4.86,4.89,4.85,4.98,4.97,4.87,4.91,f,10,0.81,255.0,42585.0,austin,austin_listings.csv,2026-04-13T02:00:43.225Z
277028,One of a kind stay In the Heart of ATX,746455,Michael,t,Neighborhood highlights,78703,null,30.28332,-97.76272,Entire home,entire home/apt,16,2.0,2 baths,5.0,13.0,"[""Private patio or balcony"", ""Stove"", ""Cleaning available during stay"", ""Cleaning products"", ""First aid kit"", ""Bathtub"", ""Wine glasses"", ""Iron"", ""Pets allowed"", ""Ceiling fan"", ""Single level home"", ""Outdoor furniture"", ""Dishwasher"", ""Hot water"", ""Shower gel"", ""Dishes and silverware"", ""Conditioner"", ""Refrigerator"", ""Freezer"", ""Self check-in"", ""Dining table"", ""Free parking on premises"", ""Keypad"", ""Wifi"", ""Outdoor dining area"", ""Heating"", ""Hair dryer"", ""Room-darkening shades"", ""Bed linens"", ""Co

In [0]:
print(df_listings_silver.columns)

['listing_id', 'name', 'host_id', 'host_name', 'host_is_superhost', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'instant_bookable', 'calculated_host_listings_count', 'reviews_per_month', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'city', 'source_file', 'ingestion_timestamp']


In [0]:
display(df_listings_silver.groupBy("city").count())

city,count
austin,10517
nashville,6633
chicago,7825
seattle,6221
twin_cities,4761


In [0]:
display(df_listings_silver.select("room_type").distinct())

room_type
entire home/apt
private room
shared room
hotel room


Step 2. Silver Calendar

Read Data

In [0]:
from pyspark.sql.functions import col, when, dayofweek

df_calendar = spark.read.format("delta").load(
    "/Volumes/workspace/default/airbnb_bronze_2/calendar"
)

In [0]:
print(df_calendar.columns)

['listing_id', 'date', 'available', 'price', 'adjusted_price', 'minimum_nights', 'maximum_nights', 'city', 'source_file', 'ingestion_timestamp']


Clean Important Columns for Analysis

In [0]:

df_calendar_silver = df_calendar.select(
    col("listing_id").cast("long").alias("listing_id"),
    col("date").cast("date").alias("calendar_date"),
    when(col("available") == "t", 1).otherwise(0).alias("is_available"),
    col("minimum_nights").cast("int").alias("calendar_minimum_nights"),
    col("maximum_nights").cast("int").alias("calendar_maximum_nights"),
    col("city"),
    col("source_file"),
    col("ingestion_timestamp")
).withColumn(
    "day_of_week",
    dayofweek(col("calendar_date"))
).withColumn(
    "is_weekend",
    when(col("day_of_week").isin([1, 7]), 1).otherwise(0)
)


Write Silver Calendar

In [0]:
df_calendar_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/airbnb_silver_2/calendar")

Validation

In [0]:
display(df_calendar_silver.limit(10))

listing_id,calendar_date,available,price,adjusted_price,calendar_minimum_nights,calendar_maximum_nights,city,source_file,ingestion_timestamp
8811549,2025-09-24,f,null,null,2,30,twin_cities,twin_cities_calendar.csv,2026-04-13T02:01:40.002Z
8811549,2025-09-25,f,null,null,2,30,twin_cities,twin_cities_calendar.csv,2026-04-13T02:01:40.002Z
8811549,2025-09-26,f,null,null,2,30,twin_cities,twin_cities_calendar.csv,2026-04-13T02:01:40.002Z
8811549,2025-09-27,f,null,null,2,30,twin_cities,twin_cities_calendar.csv,2026-04-13T02:01:40.002Z
8811549,2025-09-28,f,null,null,2,30,twin_cities,twin_cities_calendar.csv,2026-04-13T02:01:40.002Z
8811549,2025-09-29,f,null,null,2,30,twin_cities,twin_cities_calendar.csv,2026-04-13T02:01:40.002Z
8811549,2025-09-30,f,null,null,2,30,twin_cities,twin_cities_calendar.csv,2026-04-13T02:01:40.002Z
8811549,2025-10-01,f,null,null,2,30,twin_cities,twin_cities_calendar.csv,2026-04-13T02:01:40.002Z
8811549,2025-10-02,f,null,null,2,30,twin_cities,twin_cities_calendar.csv,2026-04-13T02:01:40.002Z
8811549,2025-10-03,f,null,null,2,30,twin_cities,twin_cities_calendar.csv,2026-04-13T02:01:40.002Z


In [0]:
display(df_calendar_silver.groupBy("city").count())

city,count
nashville,3446696
austin,3844547
chicago,3161995
seattle,2553540
twin_cities,1941071


In [0]:
display(df_calendar_silver.select("is_available", "day_of_week", "is_weekend").limit(10))

is_available,day_of_week,is_weekend
0,4,0
0,5,0
0,6,0
0,7,1
1,1,1
1,2,0
1,3,0
1,4,0
0,5,0
0,6,0


In [0]:
print(df_calendar_silver.columns)

['listing_id', 'calendar_date', 'is_available', 'calendar_minimum_nights', 'calendar_maximum_nights', 'city', 'source_file', 'ingestion_timestamp', 'day_of_week', 'is_weekend']


Step 3. Silver Reviews

Read Data

In [0]:
from pyspark.sql.functions import col, trim, length


df_reviews = spark.read.format("delta").load(
    "/Volumes/workspace/default/airbnb_bronze_2/reviews"
)

In [0]:
print(df_reviews.columns)

['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments', 'city', 'source_file', 'ingestion_timestamp']


Clean Important Columns For Analysis

In [0]:
df_reviews_silver = df_reviews.select(
    col("listing_id").cast("long").alias("listing_id"),
    col("id").cast("long").alias("review_id"),
    col("date").cast("date").alias("review_date"),
    col("reviewer_id").cast("long").alias("reviewer_id"),
    col("reviewer_name"),
    col("comments"),
    col("city"),
    col("source_file"),
    col("ingestion_timestamp")
).filter(
    col("comments").isNotNull()
).withColumn(
    "comments", trim(col("comments"))
).filter(
    length(col("comments")) > 0
).dropDuplicates(
    ["review_date", "reviewer_id", "comments"]
)



Write Data

In [0]:
df_reviews_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/airbnb_silver_2/reviews")

Validation

In [0]:
display(df_reviews_silver.groupBy("city").count())

city,count
nashville,784756
twin_cities,300632
seattle,575718
chicago,492377
austin,588211


In [0]:
display(df_reviews_silver.limit(10))

listing_id,review_id,review_date,reviewer_id,reviewer_name,comments,city,source_file,ingestion_timestamp
6422,47639196,2015-09-21,36087364,Kyle,"My wife and I had a wonderful, first AirBNB experience in Michele's home. She was warm and welcoming and made us feel right at home. Love the new paint job on the outside of the house! Need a picture! We were on a visit to see our daughter who is a student at Vanderbilt, and it was an easy 15 minute drive across town to the University without having to get on the interstate. Lots of good restaurants nearby and we loved having the park behind the house. We were able to enjoying exercising/walking in a beautiful, safe place. Michele (and Collier) were very enjoyable to talk to, but also were quite understanding that we needed to be able to come and go. We will definitely hope to stay there again on a return visit.",nashville,nashville_reviews.csv,2026-04-13T02:03:58.118Z
6422,560365020,2019-11-06,252101310,Sarah,"Michele and Collier really do deserve their 5 star rating as super hosts, they go above and beyond to make you feel welcome, they want you to treat their home as yours for the duration of your stay. They are non intrusive but truly welcome interaction with you, they have a wealth of local knowledge and willingly share their recommendations with you. The coffee they supply in your room is among the nicest I’ve ever had. The room is plenty big enough and there is lots of storage available so you can really settle in. The bed is so comfortable and the room a lovely temperature. The location is perfect and the house is beautiful. We cannot wait to return. Thank you so much",nashville,nashville_reviews.csv,2026-04-13T02:03:58.118Z
72906,3041445,2012-12-08,4018884,Bob,"This may be the best location in Nashville for accessibility to Broadway and the honky tonks as well as access to highways that'll take you to Franklin or any points on the compass. Street parking is available right in front of the house. You can walk to hip coffee shops, great restaurants, and more. Neighborhood is more like a park in the middle of the city, very private, quiet, and cozy but just a stone's throw from the Ryman, all kinds of live music and more. Richard was an extremely knowledgeable and resourceful host who we enjoyed very much. His laid back demeanor was refreshing, he gave us insights to different music venues, and even revealed an invaluable ""hometown"" secret that saved us parking $$$. He's also the perfect host. We're seasoned travelers, and Richard was both personable and respected our privacy. The accommodations themselves were very comfortable, spacious, and felt like home. Lots of windows, so much natural light, too. We hope to visit Nashville again sometime soon, and our first choice would be to stay with Richard again -- and again. One regret -- when Richard's home gets more known among the airbnb community, he might be too heavily booked for us to secure a reservation. That'd be bad for us, but great for Richard. Thanks, Richard, for making our Nashville getaway even better than we expected it to be! Bob and Debbie Puhala",nashville,nashville_reviews.csv,2026-04-13T02:03:58.118Z
72906,472877497,2019-06-20,31970414,Eric,A 10 minutes drive from broadway and a 5 minute walk from a bunch of local bars and restaurants richard‘s Place is located in a quiet and cozy neighborhood. The apartment on the upper floor of his lovely home is all clean and private. Richard is a very friendly guy who‘s got plenty of tips and advice where to go and make the most out of your stay. We‘d absolutely book this place again. Thank you richard for having us!,nashville,nashville_reviews.csv,2026-04-13T02:03:58.118Z
329997,17172374,2014-08-08,15641301,Rhett,Rick was a great host. Very easy to communicate with and very organized. He had outstanding instructions on what to do and how to use the space-including how to get into the place. The pictures are accurate indeed and the space was AWESOME!!! It was like your own

In [0]:
print(df_reviews_silver.columns)

['listing_id', 'review_id', 'review_date', 'reviewer_id', 'reviewer_name', 'comments', 'city', 'source_file', 'ingestion_timestamp']
